# UACB-604 · Lecture 1.1 — The genome as a string

In the lecture we treated a genome as a string over {A, C, G, T} and made predictions for **random** strings.
Here we test those predictions on a **real** genome: *Escherichia coli* K-12 MG1655 (RefSeq `NC_000913.3`, 4.64 Mb).

**You will:**
1. Download a genome straight from NCBI
2. Measure GC content and check Chargaff's second rule
3. Find the origin of replication from **GC skew**
4. Measure CpG and other dinucleotide biases
5. Test the "k-mers become unique at k ≈ log₄ n" prediction
6. Compare open-reading-frame lengths in the real genome with a shuffled genome

Run each cell with **Shift + Enter**. Cells marked ✏️ are exercises.

In [ ]:
import urllib.request, collections, random, math
import numpy as np
import matplotlib.pyplot as plt

url = ("https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
       "?db=nuccore&id=NC_000913.3&rettype=fasta&retmode=text")
fasta = urllib.request.urlopen(url).read().decode()
header, *lines = fasta.strip().split("\n")
genome = "".join(lines).upper()
print(header)
print(f"length = {len(genome):,} bp")
print(genome[:120])

## 1. Composition and Chargaff's second rule
We only downloaded **one strand**. Chargaff's first rule (A = T, G = C) is about the double helix, so it says nothing here.
The second rule claims that even a single strand has #A ≈ #T and #G ≈ #C.

In [ ]:
c = collections.Counter(genome)
n = len(genome)
for b in "ACGT":
    print(f"{b}: {c[b]:>9,}  ({c[b]/n:.4f})")
gc = (c["G"] + c["C"]) / n
print(f"\nGC content = {gc:.4f}")
print(f"A/T = {c['A']/c['T']:.4f}   G/C = {c['G']/c['C']:.4f}")

✏️ **Exercise 1.** Compute the Shannon entropy $H = -\sum_x p_x \log_2 p_x$ of the base composition. How close is it to the maximum of 2 bits? What does that say about how well simple composition-based compression could work?

In [ ]:
# your code here

## 2. GC skew finds the origin of replication
Chargaff's second rule holds for the whole chromosome but **not** for each half.
During replication the two halves of a circular chromosome are copied differently, and over time this leaves a G/C imbalance.

GC skew in a window is $\dfrac{G - C}{G + C}$. Its **cumulative sum** reaches a minimum at the origin (*oriC*) and a maximum near the terminus.

In [ ]:
w = 10_000
skew = []
for i in range(0, n - w + 1, w):
    win = genome[i:i+w]
    g, cc = win.count("G"), win.count("C")
    skew.append((g - cc) / (g + cc))
skew = np.array(skew)
cum = np.cumsum(skew)
pos = np.arange(len(skew)) * w / 1e6

fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax[0].plot(pos, skew); ax[0].axhline(0, color="grey", lw=0.5); ax[0].set_ylabel("GC skew")
ax[1].plot(pos, cum, color="C3"); ax[1].set_ylabel("cumulative skew"); ax[1].set_xlabel("position (Mb)")
print(f"min of cumulative skew at ~{pos[cum.argmin()]:.2f} Mb (predicted origin)")
print(f"max of cumulative skew at ~{pos[cum.argmax()]:.2f} Mb (predicted terminus)")
plt.show()

*oriC* of *E. coli* K-12 is annotated at about 3.93 Mb. How close did a two-line calculation get?

✏️ **Exercise 2.** Repeat with window sizes 1 kb and 100 kb. Which features of the plot are robust and which are noise?

In [ ]:
# your code here

## 3. Dinucleotides: is the genome a sequence of independent letters?
If bases were independent, the expected count of the pair $xy$ would be $N_x N_y / n$.
The ratio **observed / expected** tells us about dependence. In the lecture we saw that CpG is strongly depleted in the human genome (about 0.2–0.25). What about a bacterium?

In [ ]:
pairs = collections.Counter(genome[i:i+2] for i in range(n - 1))
oe = {}
for x in "ACGT":
    for y in "ACGT":
        oe[x+y] = pairs[x+y] / (c[x] * c[y] / n)
M = np.array([[oe[x+y] for y in "ACGT"] for x in "ACGT"])
plt.imshow(M, cmap="coolwarm", vmin=0.6, vmax=1.4)
plt.xticks(range(4), list("ACGT")); plt.yticks(range(4), list("ACGT"))
plt.xlabel("second base"); plt.ylabel("first base"); plt.colorbar(label="observed / expected")
for i in range(4):
    for j in range(4):
        plt.text(j, i, f"{M[i,j]:.2f}", ha="center", va="center")
plt.title("Dinucleotide bias in E. coli"); plt.show()

✏️ **Exercise 3.** Which dinucleotide is the most under-represented in *E. coli*? (Hint: look up "TA depletion" and why it might matter for RNA stability.) Is CpG depleted here? Why might a bacterium differ from a human? (Hint: CpG methylation.)

## 4. When do k-mers become unique?
For a random string of length $n$, the expected count of a given k-mer is $n / 4^k$, so most k-mers should be unique once $k \gtrsim \log_4 n$.
For *E. coli*, $\log_4(4.64\times10^6) \approx 11.1$. Let's check how many **distinct** k-mers appear exactly once, in the real genome and in a shuffled copy with the same composition.

In [ ]:
def frac_unique(s, k):
    cnt = collections.Counter(s[i:i+k] for i in range(len(s) - k + 1))
    once = sum(1 for v in cnt.values() if v == 1)
    return once / (len(s) - k + 1)

random.seed(1)
shuffled = "".join(random.sample(genome, len(genome)))

ks = range(6, 21, 2)
real = [frac_unique(genome, k) for k in ks]
shuf = [frac_unique(shuffled, k) for k in ks]
plt.plot(ks, real, "o-", label="E. coli")
plt.plot(ks, shuf, "s--", label="shuffled (same composition)")
plt.axvline(math.log(n, 4), color="grey", ls=":", label="log₄ n")
plt.xlabel("k"); plt.ylabel("fraction of positions whose k-mer is unique"); plt.legend(); plt.show()
for k, r, s in zip(ks, real, shuf):
    print(f"k={k:2d}  real={r:.4f}  shuffled={s:.4f}")

✏️ **Exercise 4.** At large k the real genome never quite reaches 1. Why not? (Think: rRNA operons are present in 7 copies, plus insertion sequences.) What does this mean for assembling genomes from short reads (Unit 5)?

## 5. Open reading frames: signal versus noise
In random DNA the number of codons before a stop is geometric with $p = 3/64$, mean ≈ 20.
Real genes are hundreds of codons long. Compare the run lengths in the real genome with the shuffled genome (all 3 frames, forward strand).

In [ ]:
STOPS = {"TAA", "TAG", "TGA"}
def runs_between_stops(s):
    out = []
    for f in range(3):
        run = 0
        for i in range(f, len(s) - 2, 3):
            if s[i:i+3] in STOPS:
                out.append(run); run = 0
            else:
                run += 1
    return np.array(out)

r_real = runs_between_stops(genome)
r_shuf = runs_between_stops(shuffled)
bins = np.arange(0, 600, 10)
plt.hist(r_shuf, bins=bins, alpha=0.6, label="shuffled", log=True)
plt.hist(r_real, bins=bins, alpha=0.6, label="E. coli", log=True)
plt.xlabel("codons between stops"); plt.ylabel("count (log)"); plt.legend(); plt.show()
for L in (100, 200, 300):
    print(f"runs ≥ {L} codons: real={np.sum(r_real >= L):5d}   shuffled={np.sum(r_shuf >= L):5d}")

✏️ **Exercise 5.**
1. The shuffled genome's GC content is ~0.51, not 0.5. Recompute the stop-codon probability $p$ from the actual base frequencies and compare the predicted mean with the shuffled data.
2. *E. coli* has about 4,300 annotated protein-coding genes. Roughly how many long runs (≥ 100 codons) did you find on **one strand**? What is missing from the count?
3. Write a one-paragraph answer: "Why is a long ORF evidence of a gene, and why is it not proof?"

In [ ]:
# your code here

---
**Further reading:** Brown, *Genomes 5*, ch. 1 and ch. 8 · Pevsner, *Bioinformatics and Functional Genomics*, ch. 17 (bacterial genomes).